### Building agents with OpenAI Agents SDK
- Building a basic agent and chat
- Defining and using Tools with agents
- Tracing
- Multi-Agent Orchestration (Handoffs)

#### Building a basic Agent

In [3]:
from dotenv import load_dotenv
from agents import RunConfig,OpenAIChatCompletionsModel,set_tracing_disabled
from openai import AsyncOpenAI
import os
# Load environment variables
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY environment variable is not set.")

# Set up the Gemini API-compatible client
client = AsyncOpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

model = OpenAIChatCompletionsModel(
    model="gemini-2.0-flash",
    openai_client=client
)

# Wrap into RunConfig
config = RunConfig(
    model=model,
    model_provider=client,
    tracing_disabled=True,
)

set_tracing_disabled(True)

import nest_asyncio
nest_asyncio.apply()


In [ ]:
from agents import Agent, Runner, TResponseInputItem

primary_agent = Agent(
    name="Fitness Assistant",
    instructions="You are a personal assistant that helps understand a user, their fitness profile, and their preferences and goals to provide personalized recommendations and guidance.",
    model=model,
)

result = await Runner.run(
        starting_agent=primary_agent, 
        input="Generate a fitness plan for 150lb male who wants to gain muscle and is primarily interested in going to the gym 3x per week with strength and cardio.",
        run_config=config,
)

print("Fitness Assistant:", result.final_output)


In [5]:
async def start_chat(primary_agent: Agent, chat: list[TResponseInputItem]):
   
    print("NOTE: Chat started. You can type 'EXIT' to exit the conversation.")
    print("-----------------------------------------")
    while True:

        user_input = input("You: ")
        print("User: ", user_input, "\n")
        
        if user_input == "EXIT":
            print("Fitness Assistant: Goodbye!", "\n")
            break
        
        chat.append({
            "content": user_input,
            "role" : "user",
            "type": "message"
        })
        
        result = await Runner.run(
            starting_agent=primary_agent, 
            input=chat
        )
        
        chat.clear()
        chat.extend(result.to_input_list())

        print("Fitness Assistant:", result.final_output, "\n", flush=True)

In [ ]:
chat = []
await start_chat(primary_agent, chat)

### Tools
- Provide the primary agent with the ability to compile a fitness plan

In [7]:
import json
from agents import FunctionTool, RunContextWrapper, function_tool

@function_tool
def save_markdown_report(report: str, name:str):
    """Saves the full markdown report locally for a given fitness plan.
    Use this tool to save the final report once all the details are gathered.
    You must provide the full report as a markdown string.

    Args:
        report (str): Full markdown report
        name (str): Users name
    """
    try:
        with open(f"./reports/{name}_report.md", "w") as f:
            f.write(report)
    except Exception as e:
        print(f"Failed to save report: {e}")
        raise

In [8]:
primary_agent = Agent(
    name="Fitness Assistant",
    instructions="You are a personal assistant that helps understand a user, their fitness profile, and their preferences and goals to provide personalized recommendations and guidance.",
    model=model,
    tools=[save_markdown_report]
)

chat = []
await start_chat(primary_agent, chat)

NOTE: Chat started. You can type 'EXIT' to exit the conversation.
-----------------------------------------
User:  EXIT 

Fitness Assistant: Goodbye! 



### Multi-Agent Orchestration - Handoffs
#### FITNESS AGENT
- Fitness Assistant Agent
- Workout Planner Agent
- Diet Planner Agent

In [9]:
## Using types
from pydantic import BaseModel
from enum import Enum

class UserFitnessProfile(BaseModel):
    name: str
    age: int
    weight: float
    height: float
    food_preference: str
    fitness_goal: str

class DietPlan(BaseModel):
    description: str
    target_calories: int
    protein_target: int
    breakfast: str
    lunch: str
    dinner: str

class IntensityLevel(Enum):
    Low = "Low"
    Medium = "Medium"
    High = "High"

class WorkoutPlan(BaseModel):
    description: str
    intensity: IntensityLevel
    frequency: int
    full_workout_plan: str

class Report(BaseModel):
    user_fitness_profile: UserFitnessProfile | None
    diet_plan: DietPlan | None
    workout_plan: WorkoutPlan | None


In [10]:
import json
from agents import Agent, FunctionTool, RunContextWrapper, function_tool, Runner, TResponseInputItem


@function_tool
def view_current_report(wrapper: RunContextWrapper[Report]):
    """View the current report for the User. Provides context on the UserFitnessProfile, DietPlan, and WorkoutPlan."""
    try:
        return("Current Report: ", wrapper.context.model_dump_json())
    except Exception as e:
        print(f"Failed to view user fitness profile: {e}")
        raise

@function_tool
def save_user_fitness_profile(wrapper: RunContextWrapper[Report], name:str, age:int, weight:float, height:float, food_preference:str, fitness_goal:str) -> bool:
    """Saves the user's fitness profile.

    Args:
        name (str): User's name
        age (int): User's age
        weight (float): User's weight in pounds
        height (float): User's height in inches
        food_preference (str): User's food preference (e.g., vegetarian, vegan, etc.)
        fitness_goal (str): User's fitness goal (e.g., weight loss, muscle gain, etc.)
    Returns:
        bool: True if the fitness profile was saved successfully
    Raises:
        Exception: If there was an error saving the fitness profile
    """
    
    try:
        wrapper.context.user_fitness_profile = UserFitnessProfile(
            name=name,
            age=age,
            weight=weight,
            height=height,
            food_preference=food_preference,
            fitness_goal=fitness_goal
        )
        print(f"Saved user fitness profile: {wrapper.context.user_fitness_profile.model_dump_json()}")
        return True
    except Exception as e:
        print(f"Failed to save user fitness profile: {e}")
        raise

@function_tool
def save_diet_plan(wrapper: RunContextWrapper[Report], description:str, target_calories:int, protein_target:int, breakfast:str, lunch:str, dinner:str) -> bool:
    """Saves the diet plan for the user.

    Args:
        description (str): Full description of the diet plan
        target_calories (int): Total target calories per day
        protein_target (int): Total protein target per day
        breakfast (str): Breakfast meal
        lunch (str): Lunch meal
        dinner (str): Dinner meal
        
    Returns:
        bool: True if the diet plan was saved successfully
        
    Raises:
        Exception: If there was an error saving the diet plan
    """
    try:
        wrapper.context.diet_plan = DietPlan(
            description=description,
            target_calories=target_calories,
            protein_target=protein_target,
            breakfast=breakfast,
            lunch=lunch,
            dinner=dinner
        )
        print(f"Saved diet plan: {wrapper.context.diet_plan.model_dump_json()}")
        return True
    except Exception as e:
        print(f"Failed to save diet plan: {e}")
        raise
    
@function_tool
def save_workout_plan(wrapper: RunContextWrapper[Report], description:str, intensity:IntensityLevel, frequency:int, full_workout_plan:str) -> bool:
    """Saves the workout plan for the user.

    Args:
        description (str): Description of the workout plan
        intensity (IntensityLevel): Intensity level of the workout plan
        frequency (int): Frequency of the workout plan (days per week)
        full_workout_plan (str): Complete markdown workout plan
    
    Returns
        bool: True if the workout plan was saved successfully
    
    Raises:
        Exception: If there was an error saving the workout plan
    """
    
    try:
        wrapper.context.diet_plan = WorkoutPlan(
            description=description,
            intensity=intensity,
            frequency=frequency,
            full_workout_plan=full_workout_plan
        )
        print(f"Saved workout plan: {wrapper.context.workout_plan.model_dump_json()}")
    except Exception as e:
        print(f"Failed to save diet plan: {e}")
        raise

@function_tool
def save_complete_report(wrapper: RunContextWrapper[Report], report_name:str):
    """Saves the completed report locally including the UserFitnessProfile, WorkoutPlan and DietPlan generated.
    Use this tool to save the final report once all the details are gathered.

    Args:
        report_name (str): Report file name (ex: John_Doe_Report)
    """
    try:
        with open(f"./reports/{report_name}_report.md", "w") as f:
            f.write(wrapper.context.model_dump_json())
    except Exception as e:
        print(f"Failed to save report: {e}")
        raise

async def start_chat(primary_agent: Agent, chat: list[TResponseInputItem]):
   
    print("NOTE: Chat started. You can type 'EXIT' to exit the conversation.")
    print("-----------------------------------------")
    report = Report(user_fitness_profile=None, diet_plan=None, workout_plan=None)
    while True:
        
        
        
        user_input = input("You: ")
        print("User: ", user_input, "\n")
        
        if user_input == "EXIT":
            print("Fitness Assistant: Goodbye!", "\n")
            break
        
        chat.append({
            "content": user_input,
            "role" : "user",
            "type": "message"
        })
        
        result = await Runner.run(
            starting_agent=primary_agent, 
            input=chat,
            context=report
        )
        
        chat.clear()
        chat.extend(result.to_input_list())

        print(f"{result.last_agent.name}:", result.final_output, "\n", flush=True)


In [11]:
from agents import handoff
from agents.extensions import handoff_filters


fitness_assistant = Agent(
    name="Fitness Assistant",
    instructions="You are a personal assistant that gathers information on a users fitness profile and their preferences. You begin by asking a list of questions to understand the user. Once you have the information, you save the gathered information using the tool and handover to the Workout Planner to build a workout plan. Once the workout plan is built, you handover to the Diet Planner to build a diet plan. Once both the workout and diet plans are built, you explain the plan and get the User's confirmation to save the final report using the tool.",
    model=model,
    tools=[view_current_report, save_user_fitness_profile, save_markdown_report]
)

workout_planner = Agent(
    name="Workout Planner",
    instructions="You are an expert workout planner. View the current report to retrieve the UserFitnessProfile and ask questions to build a workout plan for the User. Once you have the information, you save the gathered information using the tool and handover back to the Fitness Planner.",
    model=model,
    handoffs=[handoff(
        agent=fitness_assistant,
        input_filter=handoff_filters.remove_all_tools
    )],
    tools=[view_current_report, save_workout_plan]
)

diet_planner = Agent(
    name="Diet Planner",
    instructions="You are an expert building diet plans that tailor to a User's fitness goals. View the current report to retrieve the UserFitnessProfile and WorkoutPlan and generate a diet plan to reach the User's goals. Feel free to ask questions to the user to gather more information. Once you have the information, you save the gathered information using the tool and handover to the Fitness Assistant to explain and save the plan.",
    model=model,
    handoffs=[handoff(
        agent=fitness_assistant,
        input_filter=handoff_filters.remove_all_tools
    )],
    tools=[view_current_report, save_diet_plan]
)

fitness_assistant.handoffs = [handoff(workout_planner), handoff(diet_planner)]


In [12]:
chat = []
result = await start_chat(fitness_assistant, chat)

NOTE: Chat started. You can type 'EXIT' to exit the conversation.
-----------------------------------------
User:  Hi 

Fitness Assistant: Hello! I'm here to help you create a personalized fitness plan. To start, I need to gather some information about you. Let's begin with a few questions:

1.  What is your name?
2.  What is your age?
3.  What is your height in inches?
4.  What is your current weight in pounds?
5.  What is your primary fitness goal (e.g., weight loss, muscle gain, general fitness)?
6.  What are your food preferences (e.g., vegetarian, vegan, no dietary restrictions)?
 

User:  my name is hamza 18 5 feet 78 musclegain no dietary restrictions 

Fitness Assistant: Okay, Hamza, just to confirm, you are 18 years old, 5 feet tall, weigh 78 pounds, want to gain muscle, and have no dietary restrictions. Is that correct?
 

User:  yes 

Fitness Assistant: Okay, there seems to be a discrepancy in your height and weight. 5 feet (60 inches) and 78 pounds is underweight for an 18-